This is a brief demonstration of the workflow for the quantum protein folding problem on the FCC lattice. In the following sections, we present two key methods for building and solving the FCC Hamiltonian: polynomial fitting and the Variational Quantum Eigensolver with Constraints (VQEC) based on the Lagrangian dual method.

In [ ]:
# Necessary imports
from fcc import MiyazawaJerniganInteraction, Peptide, ProteinFoldingProblem, PenaltyParameters, ProteinSolver, ProteinFoldingResult, ProteinShapeDecoder
from fcc.measurement_utils import _evaluate_sparsepauli
import fcc
from vqe import PerturbedPrimalDualOpt
from qiskit.circuit.library import RealAmplitudes
from qiskit_aer.primitives import SamplerV2 as Sampler
from qiskit_aer.primitives import Estimator
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient
import matplotlib.pyplot as plt
import ray
import psutil
import numpy as np
from time import time

In [ ]:
# Initialize Ray for parallel processing
num_workers = psutil.cpu_count(logical=False)
print(f"Number of workers: {num_workers}")
ray.init(
    num_cpus=num_workers,
    ignore_reinit_error=True,
    log_to_driver=False,
    runtime_env={
        "py_modules": [fcc],
    },
)

## PolyFit

In [ ]:
protein_seq = "GNLVS"  # Define the amino acid sequence of the protein of interest
penalty_back = 100.0  # Backtracking penalty
penalty_redun = 100.0  # Penalty for the 4 redundant bitstrings that do not encode any of the 12 FCC directions
penalty_olap = 100.0  # Penalty for overlapping conformations

# Build the peptide object
peptide = Peptide(protein_seq)
# Set up the interaction and penalty terms
mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file="mj_matrix")
penalty_terms = PenaltyParameters(
    penalty_back=penalty_back, penalty_redun=penalty_redun, penalty_olap=penalty_olap
)
# Build the protein folding problem
pf_problem = ProteinFoldingProblem(
    peptide=peptide, interaction=mj_interaction, penalty_parameters=penalty_terms
)
# Get the Hamiltonian
qubit_op = pf_problem.qubit_op(r2_threshold=1.0, chunk=20)
print(f"Number of qubits: {qubit_op.num_qubits}")
print(f"Number of terms in the Hamiltonian: {qubit_op.size}")

In [ ]:
# Choose the ansatz for the VQE
ansatz = RealAmplitudes(qubit_op.num_qubits, reps=1, entanglement="linear").decompose()
# Set up the sampler
shots = 10_000
sampler = Sampler(default_shots=shots)
# Set up the solver
optimizer = "COBYLA"
max_iter = 200
num_batches = num_workers  # Parallelize the energy evaluations
protein_solver = ProteinSolver(
    ansatz=ansatz,
    hamiltonian=qubit_op,
    sampler=sampler,
    parallelizer="ray",  # Use Ray for parallel processing
)
# Run the VQE
start_time = time()
polyfit_result = protein_solver.train(
    optimizer=optimizer,
    maxiter=max_iter,
    num_batches=num_batches,
)
end_time = time()
print(f"VQE completed in {end_time - start_time:.2f} seconds")
print(f"VQE result keys: {polyfit_result.keys()}")

In [ ]:
polyfit_result["top_solutions"][:5]

In [ ]:
# Plot the top 5 solutions
for i, (bitstring, energy) in enumerate(polyfit_result["top_solutions"][:5]):
    pf_result = ProteinFoldingResult(
        peptide=peptide,
        unused_qubits=pf_problem.unused_qubits,
        solution_bitstring=bitstring,
    )
    fig = pf_result.get_figure(
        title=f"Best solution {i+1}: {bitstring} (Energy: {energy:.2f})"
    )

## VQEC

The VQEC workflow is set up in a similar way, except that we don't explicitly provide the overlap penalty term such that the corresponding term is not included in the Hamiltonian. Instead, we have to build separate qubit operators for these constraints.

In [ ]:
protein_seq = "GNLVS"  # Define the amino acid sequence of the protein of interest
penalty_back = 100.0  # Backtracking penalty
penalty_redun = 100.0  # Penalty for the 4 redundant bitstrings that do not encode any of the 12 FCC directions

# Build the peptide object
peptide = Peptide(protein_seq)
# Set up the interaction and penalty terms
mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file="mj_matrix")
penalty_terms = PenaltyParameters(
    penalty_back=penalty_back, penalty_redun=penalty_redun
)
# Build the protein folding problem
pf_problem = ProteinFoldingProblem(
    peptide=peptide, interaction=mj_interaction, penalty_parameters=penalty_terms
)
# Get the Hamiltonian
qubit_op = pf_problem.qubit_op(r2_threshold=1.0, chunk=20)
print(f"Number of qubits: {qubit_op.num_qubits}")
print(f"Number of terms in the Hamiltonian: {qubit_op.size}")
# Build the constraint operators
bead_pairs, olap_constr_ops = pf_problem.olap_constr_ops()
print(f"Number of overlap constraints: {len(olap_constr_ops)}")

In [ ]:
# Use RealAmplitudes ansatz
ansatz = RealAmplitudes(qubit_op.num_qubits, reps=1, entanglement="linear").decompose()
# Set up the estimator and gradient estimator
estimator = Estimator(
    backend_options={"method": "statevector"},
    run_options={"shots": None},
    approximation=True,
)
gradient = ParamShiftEstimatorGradient(estimator=estimator)

# Hyperparameters for the VQEC optimization
init_dual_vars = np.array([0.0] * len(olap_constr_ops))
perturb_step = 0.05
gamma = 0.5
max_iter = 100

vqec_solver = PerturbedPrimalDualOpt(
    qubit_op=qubit_op,
    constr_ops=olap_constr_ops,
    ansatz=ansatz,
    estimator=estimator,
    gradient=gradient,
)
vqec_result = vqec_solver.optimize_primal_dual(
    initial_dual_vars=init_dual_vars,
    primal_perturb_step=perturb_step,
    dual_perturb_step=perturb_step,
    gamma=gamma,
    auto_update_step=False,
    max_iter=max_iter,
)

In [ ]:
vqec_result.keys()

In [ ]:
# Plot the energy and constraint expectations
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(vqec_result["energy_history"], label="Energy", marker=".")
ax[0].set_xlabel("Iteration")
ax[0].set_ylabel("Energy expectation")

for i, constr_op in enumerate(olap_constr_ops):
    ax[1].plot(
        np.array(vqec_result["constraints_history"])[:, i],
        label=f"Constraint {i}",
        marker=".",
    )
ax[1].set_xlabel("Iteration")
ax[1].set_ylabel("Constraint expectation")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# With the optimal primal variables (i.e., circuit parameters), we can sample from the optimized circuit
shots = 10_000  # Number of shots for sampling
sampler = Sampler(default_shots=shots)
ansatz.measure_all()  # Ensure the ansatz has measurement operations
pub = (ansatz, vqec_result["optimal_primal_vars"])
sampler_result = sampler.run([pub]).result()

# Get the counts
counts = sampler_result[0].data.meas.get_counts()

# Sort the binary strings by their counts
sorted_quasi_dist = dict(sorted(counts.items(), key=lambda item: item[1], reverse=True))
# Compute the energies of all samples
bitstring_energies = {}
for i, (sample, count) in enumerate(sorted_quasi_dist.items()):
    if i >= 5:
        break
    energy = _evaluate_sparsepauli(sample, qubit_op)
    bitstring_energies[sample] = energy
    print(f"Sample {i}: {sample} with probability {count / shots} and energy {energy}")

In [ ]:
for i, (sample, energy) in enumerate(bitstring_energies.items()):
    pf_result = ProteinFoldingResult(
        peptide=peptide,
        unused_qubits=pf_problem.unused_qubits,
        solution_bitstring=sample,
    )
    fig = pf_result.get_figure(
        title=f"Most frequent solution {i+1}: {sample} (Energy: {energy:.2f})"
    )
    plt.show()

## Comparison to classical exhaustive search results

In [ ]:
from fcc.classical_utils import load_top_cls_solns

file_name = "topobj_GNLVS.txt"
top_cls_solutions = load_top_cls_solns(file_name)
top_cls_solutions[:5]

In [ ]:
# Example: Translate the top polyfit solutions to turn sequences
top_polyfit_solutions = polyfit_result["top_solutions"][:5]
print(top_polyfit_solutions)

polyfit_turns = []
for i, (bitstring, energy) in enumerate(top_polyfit_solutions):
    pf_decoder = ProteinShapeDecoder(
        peptide=peptide,
        solution_bitstring=bitstring,
    )
    polyfit_turns.append([pf_decoder.turn_sequence, energy])

polyfit_turns